# model_experiment_NBEATS

N-BEATS (Oreshkin et al. 2020) via `neuralforecast`: deep stacks of fully-connected blocks, each producing a **backcast** (what it explains of the input) and a **forecast**; later blocks work on the residual the earlier ones couldn't explain. Two variants:
- **generic** — identity basis, pure learned blocks;
- **interpretable** — blocks constrained to polynomial **trend** + Fourier **seasonality** bases, so the model's decomposition is human-readable.

Like PatchTST: univariate, no exogenous features, plain MAE loss (the 5x holiday weight can't be injected — same documented limitation).

**Kaggle settings: Accelerator = GPU T4, Internet = On.** MLflow experiment: **NBEATS_Training**.

In [ ]:
# Kaggle bootstrap — attach competition data + the three MLFLOW secrets first.
import os
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    os.system("pip install -q mlflow neuralforecast")
    if not os.path.exists("walmart-sales-forecasting"):
        os.system("git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git")
    import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")
    import glob, shutil, zipfile
    src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
    assert src, "competition data not attached"
    os.makedirs("data", exist_ok=True)
    for f in glob.glob(src[0] + "/*"):
        (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
    os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

import torch
print("GPU available:", torch.cuda.is_available())

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import pandas as pd
import mlflow

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow
from src.postprocess import naive_lag52, apply_christmas_shift, blend_holiday_naive
from src.preprocessing import BLEND_HOLIDAY_WEEKS

train, test, features, stores = load_raw("data" if ON_KAGGLE else None)
setup_mlflow("NBEATS_Training")

GPU_KW = ({"accelerator": "gpu", "devices": 1} if torch.cuda.is_available()
          else {"accelerator": "cpu"})


def to_nf(df):
    out = df[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
    out["unique_id"] = out["Store"].astype(str) + "_" + out["Dept"].astype(str)
    return out.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["unique_id", "ds", "y"]]


def from_nf(fc, model_col):
    out = fc.reset_index() if "unique_id" not in fc.columns else fc.copy()
    sd = out["unique_id"].str.split("_", expand=True).astype(int)
    out["Store"], out["Dept"] = sd[0], sd[1]
    return out.rename(columns={"ds": "Date", model_col: "pred"})[["Store", "Dept", "Date", "pred"]]

## Run 1 — Fold-1 evaluation: generic vs interpretable stacks

In [ ]:
def eval_fold1(stack_types, max_steps=1500, input_size=52, seed=42):
    tr, va = split_fold(train, 1)
    model = NBEATS(h=39, input_size=input_size,
                   stack_types=stack_types,
                   loss=MAE(), max_steps=max_steps, batch_size=64,
                   windows_batch_size=512, random_seed=seed,
                   start_padding_enabled=True, **GPU_KW)
    nf = NeuralForecast(models=[model], freq="W-FRI")
    nf.fit(to_nf(tr))
    fc = from_nf(nf.predict(), "NBEATS")
    m = va.merge(fc, on=["Store", "Dept", "Date"], how="left")
    coverage = m["pred"].notna().mean()
    m["pred"] = m["pred"].fillna(pd.Series(naive_lag52(tr, va), index=m.index))
    dep_med = tr.groupby("Dept")["Weekly_Sales"].median()
    m["pred"] = m["pred"].fillna(m["Dept"].map(dep_med)).fillna(0)
    rep = wmae_report(m["Weekly_Sales"], m["pred"], m["IsHoliday"])
    return rep, coverage, m, tr

VARIANTS = {
    "generic": ["identity", "identity", "identity"],
    "interpretable": ["trend", "seasonality"],
}
results = {}
for name, stacks in VARIANTS.items():
    rep, cov, m1, tr1 = eval_fold1(stacks)
    results[name] = rep["wmae"]
    with mlflow.start_run(run_name=f"NBEATS_CV_{name}"):
        mlflow.log_params({"stack_types": str(stacks), "input_size": 52,
                           "max_steps": 1500, "fold": 1})
        mlflow.log_metrics({**rep, "coverage": cov})
    print(f"{name}: coverage {cov:.3f}, {rep}")

BEST_VARIANT = min(results, key=results.get)
print("best variant:", BEST_VARIANT)

## Run 2 — Holiday blend (no Christmas; rationale in the LightGBM notebook)

In [ ]:
rep1, _, m1, tr1 = eval_fold1(VARIANTS[BEST_VARIANT])
blend_scores = {}
for w in (0.0, 0.5, 0.75):
    blended = blend_holiday_naive(m1[["Store", "Dept", "Date", "pred"]], tr1,
                                  weight=w, holiday_dates=BLEND_HOLIDAY_WEEKS)
    rep = wmae_report(m1["Weekly_Sales"], blended["pred"], m1["IsHoliday"])
    blend_scores[w] = rep["wmae"]
    with mlflow.start_run(run_name=f"NBEATS_Blend_noXmas_w{w}"):
        mlflow.log_params({"variant": BEST_VARIANT, "blend_weight": w, "fold": 1})
        mlflow.log_metrics(rep)
    print(f"w={w}: {rep}")
BLEND_W = min(blend_scores, key=blend_scores.get)
print("best blend weight:", BLEND_W)

## Run 3 — Final: train on ALL data (input 104), predict test

In [ ]:
final_model = NBEATS(h=39, input_size=104,
                     stack_types=VARIANTS[BEST_VARIANT],
                     loss=MAE(), max_steps=3000, batch_size=64,
                     windows_batch_size=512, random_seed=42,
                     start_padding_enabled=True, **GPU_KW)
nf = NeuralForecast(models=[final_model], freq="W-FRI")
nf.fit(to_nf(train))

fc = from_nf(nf.predict(), "NBEATS")
sub = test[["Store", "Dept", "Date"]].merge(fc, on=["Store", "Dept", "Date"], how="left")
covered = sub["pred"].notna().mean()
sub["pred"] = sub["pred"].fillna(pd.Series(naive_lag52(train, test), index=sub.index))
dep_med = train.groupby("Dept")["Weekly_Sales"].median()
sub["pred"] = sub["pred"].fillna(sub["Dept"].map(dep_med)).fillna(0)

sub = apply_christmas_shift(sub, pred_col="pred")
if BLEND_W > 0:
    sub = blend_holiday_naive(sub, train, weight=BLEND_W, holiday_dates=BLEND_HOLIDAY_WEEKS)

with mlflow.start_run(run_name="NBEATS_Final"):
    mlflow.log_params({"variant": BEST_VARIANT, "input_size": 104, "max_steps": 3000,
                       "blend_weight": BLEND_W, "post": "christmas_shift + noXmas_blend"})
    mlflow.log_metrics({"fold1_wmae": results[BEST_VARIANT], "test_coverage": covered})
    nf.save(path="nbeats_model", overwrite=True)
    mlflow.log_artifacts("nbeats_model", artifact_path="model")

make_submission(sub, "pred", "submission_nbeats.csv")
print(f"wrote submission_nbeats.csv (coverage {covered:.3f}, blend w={BLEND_W})")